In [0]:
data = [
    (1, "A", 5000),
    (2, "B", 6000),
    (3, "C", 7000)
]

columns = ["id", "name", "salary"]
from delta.tables import *
df = spark.createDataFrame(data, columns)
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save("/Volumes/workspace/default/practice/delta")

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/practice/delta")
df.display()

id,name,salary
1,A,5000
2,B,6000
3,C,7000


In [0]:
new_data=[(4,"E",8000)]
new_df=spark.createDataFrame(new_data,columns)
new_df.write.format("delta").mode("append").save("/Volumes/workspace/default/practice/delta")
new_df=spark.read.format("delta").load("/Volumes/workspace/default/practice/delta")
new_df.display()


id,name,salary
1,A,5000
2,B,6000
3,C,7000
4,E,8000


In [0]:
update_table=DeltaTable.forPath(spark,"/Volumes/workspace/default/practice/delta")
update_table.update(
    condition="name='E'",
    set={"salary":"8700"}
)
df = spark.read.format("delta").load("/Volumes/workspace/default/practice/delta")
df.show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  1|   A|  5000|
|  2|   B|  6000|
|  3|   C|  7000|
|  4|   E|  8700|
+---+----+------+



In [0]:
update_table.delete("name='A'")
df = spark.read.format("delta").load("/Volumes/workspace/default/practice/delta")
df.show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  2|   B|  6000|
|  3|   C|  7000|
|  4|   E|  8700|
+---+----+------+



In [0]:
updates = [
    (2, "B", 7000),  # update
    (5, "E", 9000)   # insert
]

updates_df = spark.createDataFrame(updates, columns)
updates_df.display()

id,name,salary
2,B,7000
5,E,9000


In [0]:
update_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "salary": "source.salary"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "salary": "source.salary"
}).execute()
df = spark.read.format("delta").load("/Volumes/workspace/default/practice/delta")
df.show()

+---+----+------+
| id|name|salary|
+---+----+------+
|  3|   C|  7000|
|  4|   E|  8700|
|  2|   B|  7000|
|  5|   E|  9000|
+---+----+------+



In [0]:
new_data = [(6, "F", 10000, "India")]
new_columns = ["id", "name", "salary", "country"]
df_new = spark.createDataFrame(new_data, new_columns)
df_new.display()
df_new.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/Volumes/workspace/default/practice/delta")
df_read=spark.read.format('delta').load("/Volumes/workspace/default/practice/delta")
df_read.display()

id,name,salary,country
6,F,10000,India


id,name,salary,country
6,F,10000,India
3,C,7000,null
4,E,8700,null
2,B,7000,null
5,E,9000,null


##Time travel

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark, "/Volumes/workspace/default/practice/delta"
)

delta_table.history().show()

+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|     27|2026-04-14 14:32:25|73109523991213|keerthisrivalli18...|    WRITE|{mode -> Append, ...|NULL|{3904375379557492}|   e122794c-3cd2-444...|0414-133610-mbo1j...|         26|WriteSerializable|         t

In [0]:
df_old = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load("/Volumes/workspace/default/practice/delta")
df_old = DeltaTable.forPath(
    spark, "/Volumes/workspace/default/practice/delta"
)
df_old.history().show(100)

+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|        userId|            userName|operation| operationParameters| job|          notebook|queryHistoryStatementId|           clusterId|readVersion|   isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+--------------+--------------------+---------+--------------------+----+------------------+-----------------------+--------------------+-----------+-----------------+-------------+--------------------+------------+--------------------+
|     27|2026-04-14 14:32:25|73109523991213|keerthisrivalli18...|    WRITE|{mode -> Append, ...|NULL|{3904375379557492}|   e122794c-3cd2-444...|0414-133610-mbo1j...|         26|WriteSerializable|         t

In [0]:
df_old.restoreToVersion(0).display()

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1043,1,2,1,2308,1043
